In [1]:
import os
import numpy as np
import random
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

import monai
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd, Spacingd, Resized, RandFlipd,
    RandRotate90d, RandShiftIntensityd, ToTensord, EnsureTyped
)
from monai.data import Dataset, CacheDataset, decollate_batch
from monai.networks.nets import UNet, SwinUNETR
from monai.losses import DiceLoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.transforms import AsDiscrete
from monai.inferers import sliding_window_inference

import SimpleITK as sitk  # used in your notebook earlier
import joblib
import time

Matplotlib is building the font cache; this may take a moment.


In [7]:
import glob
import pandas as pd

save_dir = "csv_lists"

images_dir = "Task07_Pancreas/imagesTr"
labels_dir = "Task07_Pancreas/labelsTr"
test_dir = "Task07_Pancreas/imagesTs"

os.makedirs(save_dir, exist_ok=True)

In [8]:
def make_train_val_csv(images_dir, labels_dir, split_ratio=0.8):

    images = sorted([
        f for f in glob.glob(os.path.join(images_dir, "*.nii"))
        if not os.path.basename(f).startswith("._")
    ])

    labels = sorted([
        f for f in glob.glob(os.path.join(labels_dir, "*.nii"))
        if not os.path.basename(f).startswith("._")
    ])

    data = list(zip(images, labels))

    random.shuffle(data)

    split = int(len(data) * split_ratio)

    train = data[:split]
    val = data[split:]

    pd.DataFrame(train, columns=["image","label"]).to_csv(
        os.path.join(save_dir,"train.csv"),
        index=False
    )

    pd.DataFrame(val, columns=["image","label"]).to_csv(
        os.path.join(save_dir,"val.csv"),
        index=False
    )

    print("Train =", len(train))
    print("Validation =", len(val))

In [9]:
def make_test_csv(test_dir):

    test = sorted([
        f for f in glob.glob(os.path.join(test_dir, "*.nii"))
        if not os.path.basename(f).startswith("._")
    ])

    pd.DataFrame(test, columns=["image"]).to_csv(
        os.path.join(save_dir,"test.csv"),
        index=False
    )

    print("Test =", len(test))

In [10]:
make_train_val_csv(images_dir, labels_dir)
make_test_csv(test_dir)

Train = 224
Validation = 57
Test = 139


In [11]:
import pandas as pd

train_df = pd.read_csv("csv_lists/train.csv")
val_df = pd.read_csv("csv_lists/val.csv")
test_df = pd.read_csv("csv_lists/test.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

train_df.head()

Train: 224
Validation: 57
Test: 139


,image,label
0,Task07_Pancreas/imagesTr\pancreas_262.nii,Task07_Pancreas/labelsTr\pancreas_262.nii
1,Task07_Pancreas/imagesTr\pancreas_286.nii,Task07_Pancreas/labelsTr\pancreas_286.nii
2,Task07_Pancreas/imagesTr\pancreas_411.nii,Task07_Pancreas/labelsTr\pancreas_411.nii
3,Task07_Pancreas/imagesTr\pancreas_035.nii,Task07_Pancreas/labelsTr\pancreas_035.nii
4,Task07_Pancreas/imagesTr\pancreas_329.nii,Task07_Pancreas/labelsTr\pancreas_329.nii


In [12]:
#converting the dataframes to lists of dictionaries for MONAI Dataset
train_files = [
    {"image": img, "label": lbl}
    for img, lbl in zip(train_df["image"], train_df["label"])
]

val_files = [
    {"image": img, "label": lbl}
    for img, lbl in zip(val_df["image"], val_df["label"])
]

test_files = [
    {"image": img}
    for img in test_df["image"]
]

print(train_files[0])

{'image': 'Task07_Pancreas/imagesTr\\pancreas_262.nii', 'label': 'Task07_Pancreas/labelsTr\\pancreas_262.nii'}


In [13]:
print("Train samples :", len(train_files))
print("Validation samples :", len(val_files))
print("Test samples :", len(test_files))

Train samples : 224
Validation samples : 57
Test samples : 139


In [14]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityd,
    CropForegroundd,
    Orientationd,
    Spacingd,
    ResizeD,
    RandFlipd,
    RandRotate90d,
    EnsureTyped
)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),

    EnsureChannelFirstd(keys=["image", "label"]),

    Orientationd(
        keys=["image", "label"],
        axcodes="RAS"
    ),

    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest")
    ),

    ScaleIntensityd(keys=["image"]),

    CropForegroundd(
        keys=["image", "label"],
        source_key="image"
    ),

    ResizeD(
        keys=["image", "label"],
        spatial_size=(128,128,128)
    ),

    RandFlipd(
        keys=["image","label"],
        prob=0.5,
        spatial_axis=0
    ),

    RandRotate90d(
        keys=["image","label"],
        prob=0.5,
        max_k=3
    ),

    EnsureTyped(keys=["image","label"])
])

c:\Pancreas_\.venv\Lib\site-packages\monai\utils\deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [ ]:
#validation transforms
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),

    EnsureChannelFirstd(keys=["image", "label"]),

    Orientationd(
        keys=["image", "label"],
        axcodes="RAS"
    ),

    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5,1.5,2.0),
        mode=("bilinear","nearest")
    ),

    ScaleIntensityd(keys=["image"]),

    CropForegroundd(
        keys=["image","label"],
        source_key="image"
    ),

    ResizeD(
        keys=["image","label"],
        spatial_size=(128,128,128)
    ),

    EnsureTyped(keys=["image","label"])
])

In [16]:
test_transforms = Compose([
    LoadImaged(keys=["image"]),

    EnsureChannelFirstd(keys=["image"]),

    Orientationd(
        keys=["image"],
        axcodes="RAS"
    ),

    Spacingd(
        keys=["image"],
        pixdim=(1.5,1.5,2.0),
        mode="bilinear"
    ),

    ScaleIntensityd(keys=["image"]),

    CropForegroundd(
        keys=["image"],
        source_key="image"
    ),

    ResizeD(
        keys=["image"],
        spatial_size=(128,128,128)
    ),

    EnsureTyped(keys=["image"])
])